<a href="https://colab.research.google.com/github/threaddotsleep/senilebot/blob/main/senile_bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q torch transformers accelerate bitsandbytes gradio

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

model_id = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

# Configure 4-bit quantization with CPU offloading enabled inside the config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    llm_int8_enable_fp32_cpu_offload=True
)

# Load model with the updated configuration
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [4]:
import gradio as gr

def predict(message, history):
    # System prompt shaping the persona
    system_prompt = (
        "You are an elderly character experiencing mild-to-moderate cognitive decline. "
        "Mix up timelines, confuse modern things with mid-20th-century life, lose your train "
        "of thought mid-sentence, and occasionally pivot to unrelated memories or ask repetitive questions. "
        "Never break character."
    )

    messages = [{"role": "system", "content": system_prompt}]

    # Map Gradio history format into the model's chat template
    for human_msg, assistant_msg in history:
        messages.append({"role": "user", "content": human_msg})
        messages.append({"role": "assistant", "content": assistant_msg})

    # Append the current user prompt
    messages.append({"role": "user", "content": message})

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    outputs = pipe(
        prompt,
        max_new_tokens=150,
        do_sample=True,
        temperature=0.85,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id
    )

    response = outputs[0]['generated_text'][len(prompt):].strip()
    return response

In [ ]:
with gr.Blocks(theme="soft") as demo:
    gr.ChatInterface(
        fn=predict,
        title="A Gentle Conversation",
        description="A chat persona exploring memory, time, and meandering stories."
    )

# share=True creates a public gradio.live link running directly from your Colab GPU session
demo.launch(share=True, debug=True)

/tmp/ipykernel_814/2975458675.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme="soft") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f026754e3a35cd1bb3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
